# Training diagnostics

Compare the matched 2B runs.

In [ ]:
import json
from pathlib import Path

from IPython.display import Image, display

from utils.extract_training_curves import extract_training_curves
from utils.plot_training_curves import load_curves, plot_scalar

repo_root = Path.cwd()
if not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent

runs_root = repo_root / "outputs/finetuning"
curves_path = repo_root / "notebooks/analysis/training_curves.csv"
figures_dir = repo_root / "notebooks/analysis/figures"

initial_runs = {
    "no_loc": "11329",
    "loc_text": "11330",
    "loc_embed": "11328",
}
full_runs = {
    "no_loc": "11270",
    "loc_text": "11271",
    "loc_embed": "11272",
}
initial_jobs = set(initial_runs.values())
full_jobs = set(full_runs.values())

extract_training_curves(
    runs_root=runs_root,
    output=curves_path,
    jobs=initial_jobs | full_jobs,
)
curves = load_curves(curves_path)

def plot_and_show(**kwargs):
    plot_scalar(**kwargs)
    display(Image(filename=str(kwargs["output"])))

## Initial convergence diagnostic

In [ ]:
plot_and_show(
    df=curves,
    tag="val/loss",
    jobs=initial_jobs,
    output=figures_dir / "initial_2b_validation_loss.png",
    title="2B initial convergence diagnostic",
    ylabel="Validation loss (fixed 8,192-row subset)",
)

In [ ]:
plot_and_show(
    df=curves,
    tag="train/loss_step",
    jobs=initial_jobs,
    output=figures_dir / "initial_2b_training_loss.png",
    title="2B initial convergence diagnostic",
    ylabel="Training loss",
)

## Full runs

Validation uses the same fixed 8,192-row subset in all three runs; it is not the full validation split or the final benchmark.

In [ ]:
plot_and_show(
    df=curves,
    tag="val/loss",
    jobs=full_jobs,
    output=figures_dir / "full_2b_validation_loss.png",
    title="2B full finetuning runs",
    ylabel="Validation loss (fixed 8,192-row subset)",
)

In [ ]:
plot_and_show(
    df=curves,
    tag="train/loss_step",
    jobs=full_jobs,
    output=figures_dir / "full_2b_training_loss.png",
    title="2B full finetuning runs",
    ylabel="Training loss",
)

In [ ]:
plot_and_show(
    df=curves,
    tag="train/loss_step",
    jobs=full_jobs,
    max_step=1_000,
    output=figures_dir / "full_2b_training_loss_first_1000_steps.png",
    title="2B full finetuning: first 1,000 steps",
    ylabel="Training loss",
)

In [ ]:
plot_and_show(
    df=curves,
    tag="train/loss_step",
    jobs=full_jobs,
    min_step=500,
    output=figures_dir / "full_2b_training_loss_after_500_steps.png",
    title="2B full finetuning: after step 500",
    ylabel="Training loss",
)

## Qualitative examples from the initial diagnostic

Show the same samples at the first validation check, steps 20, 40, 60 and 80, and the final check. `Adjacency` means that two land-cover regions touch or share a boundary.

In [ ]:
examples = {}
for condition, job_id in initial_runs.items():
    path = runs_root / job_id / "validation_generations.jsonl"
    if not path.exists():
        print(f"Missing {condition} generations: {path}")
        continue
    rows = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line]
    examples[condition] = {
        (row["global_step"], str(row["sample_id"])): row for row in rows
    }

if examples:
    common_steps = sorted(set.intersection(*[{step for step, _ in rows} for rows in examples.values()]))
    requested_steps = [common_steps[0], 20, 40, 60, 80, common_steps[-1]]
    selected_steps = list(dict.fromkeys(step for step in requested_steps if step in common_steps))

    for step in selected_steps:
        label = "first validation check" if step == common_steps[0] else "final validation check" if step == common_steps[-1] else f"step {step}"
        print(f"\n{'#' * 36} {label} {'#' * 36}")
        sample_ids = sorted({sample_id for rows in examples.values() for row_step, sample_id in rows if row_step == step})
        for sample_id in sample_ids:
            example = next(rows[(step, sample_id)] for rows in examples.values() if (step, sample_id) in rows)
            print("=" * 90)
            print(f"{example.get('task_type')} — {example.get('task_category')} (sample {sample_id})")
            print(f"Prompt: {example.get('input_text')}")
            print(f"Reference: {'; '.join(example.get('target_texts', []))}")
            for condition in initial_runs:
                row = examples.get(condition, {}).get((step, sample_id))
                prediction = row.get("prediction") if row else "[missing]"
                print(f"{condition:9}: {prediction}")

## Qualitative evolution by sample

Follow each location mode across the selected validation checks.

In [ ]:
sample_ids = sorted({sample_id for rows in examples.values() for _, sample_id in rows})
for sample_id in sample_ids:
    example = next(
        row
        for rows in examples.values()
        for (step, row_sample_id), row in rows.items()
        if row_sample_id == sample_id and step in selected_steps
    )
    print("=" * 100)
    print(f"{example.get('task_type')} — {example.get('task_category')} (sample {sample_id})")
    print(f"Prompt: {example.get('input_text')}")

    for condition in initial_runs:
        print(f"\n{condition}")
        for step in selected_steps:
            row = examples.get(condition, {}).get((step, sample_id))
            prediction = row.get("prediction") if row else "[missing]"
            print(f"  step {step:>3}: {prediction}")
        print(f"  reference: {'; '.join(example.get('target_texts', []))}")
    print()